# Lecture 5.8: Input Guardrails — Validating User Messages Before the Run

In Lecture 4.6 you saw the shape of `InputGuardrailTripwireTriggered` without
building the guardrail that raises it. In this lecture you build the real
thing: a check that runs on the user's message before your agent commits to
answering it.

By the end of this notebook you will be able to:

- Write an input guardrail with the `@input_guardrail` decorator
- Use an entire sub-agent as the check, reading a typed classification off it
- Control **when** the guardrail runs relative to the main agent with `run_in_parallel`
- Attach guardrails at the run level with `RunConfig.input_guardrails`
- Read an `InputGuardrailResult` after a tripwire fires

This notebook uses the Google Colab Secrets method for API key setup and pins
the `openai-agents` package to a fixed version for reproducibility.

## Cell 1: Install the OpenAI Agents SDK

This installs the `openai-agents` package, which provides the `Agent`,
`Runner`, and guardrail classes used throughout this notebook.

The version is pinned below for reproducibility. If the package is already
present in this Colab session (for example, if you already ran an earlier
cell in this notebook), this command completes almost instantly since pip
detects the exact version is already satisfied.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00


## Cell 2: Configure Your OpenAI API Key

This cell reads your OpenAI API key out of Colab Secrets and writes it to the
`OPENAI_API_KEY` environment variable, which the SDK reads automatically.

**To add the secret in Colab:**

1. Click the key icon (🔑) in the left sidebar to open the Secrets panel.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.

**If you're running locally instead of in Colab:** set the environment
variable in your terminal before launching Jupyter, for example
`export OPENAI_API_KEY="sk-..."` on macOS or Linux, and skip this cell.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Set the Model Name

`MODEL_NAME` is declared once here and reused for every agent in this
notebook. Changing this single variable updates the model used everywhere
below, so you never have to hunt down a hardcoded model string.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"


## Cell 4: Import Guardrail Components

| Import | What it's for |
|---|---|
| `time` | Measures elapsed seconds for the parallel-vs-blocking timing demo later in this notebook |
| `BaseModel` (pydantic) | Defines the typed output shape for the guardrail's classification agent |
| `Reasoning` | Configures GPT-5 reasoning effort on `ModelSettings`, imported from `openai.types.shared` |
| `Agent` | Defines both the classification agent used inside the guardrail and the main agents it protects |
| `GuardrailFunctionOutput` | The dataclass a guardrail function must return: `output_info` plus `tripwire_triggered` |
| `InputGuardrailTripwireTriggered` | The exception raised when a guardrail's tripwire fires, first previewed in Lecture 4.6 |
| `ModelSettings` | Passed to `Agent(model_settings=...)` to configure reasoning effort and verbosity |
| `RunConfig` | Passed to `Runner.run(run_config=...)`; used later in this notebook to attach a guardrail at the run level |
| `RunContextWrapper` | The first parameter every guardrail function receives, giving access to the run's context |
| `Runner` | Runs agents with `await Runner.run(...)` |
| `TResponseInputItem` | The typed input-item shape used in guardrail function signatures |
| `input_guardrail` | The decorator that turns a plain function into an `InputGuardrail` |

`input_guardrail` is a decorator imported directly from the top-level `agents`
package, alongside the other guardrail classes.

In [4]:
import time

from pydantic import BaseModel
from openai.types.shared import Reasoning
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    ModelSettings,
    RunConfig,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)

## What Input Guardrails Do

Input guardrails are checks that run either in parallel with the agent or
before it starts. They can be used to do things like:

- Check if input messages are off-topic
- Take over control of the agent's execution if an unexpected input is detected

If `result.tripwire_triggered` is `True`, the agent's execution will
immediately stop, and an `InputGuardrailTripwireTriggered` exception will be
raised.

That last point is the mechanism the rest of this notebook is built around.
A guardrail doesn't return an error code you have to check yourself. It
raises an exception, and you catch it exactly like any other Python
exception.

## Cell 5: The Canonical Example — Math Homework Guardrail

This cell defines the guardrail from the ground up, in three pieces.

**`MathHomeworkOutput`** is a Pydantic model with two fields:
`is_math_homework` (the boolean the guardrail cares about) and `reasoning`
(a short explanation, useful for debugging and for showing in the UI if a
request gets blocked).

**`guardrail_agent`** is a small, cheap agent whose only job is
classification. Its `output_type` is set to `MathHomeworkOutput`, so every
run of this agent returns a structured object instead of free text. Notice
its `model_settings`: `reasoning=Reasoning(effort="none")` and
`verbosity="low"` keep this classification call fast, since there's no need
for deep reasoning to decide whether a message is a math question.

**`math_guardrail`** is the guardrail function itself, decorated with
`@input_guardrail`. Its signature is fixed by the SDK: it receives a
`RunContextWrapper`, the `Agent` being protected, and the input (a string or
list of input items). Inside, it runs `guardrail_agent` on that same input
and reads `result.final_output.is_math_homework` off the typed result. It
returns a `GuardrailFunctionOutput`, where `output_info` carries the full
classification (useful for later inspection) and `tripwire_triggered` is the
boolean that decides whether execution halts.

**`support_agent`** is the actual agent your users talk to. It attaches the
guardrail through `input_guardrails=[math_guardrail]`. This is a list, so an
agent can carry more than one input guardrail at once, which you'll see
later in this notebook.

| Parameter | Type | Role |
|---|---|---|
| `guardrail_function` | Callable | The decorated function itself; the SDK wires this up automatically |
| `name` | `str \| None` | Optional name for tracing; defaults to the function's `__name__` if omitted |
| `run_in_parallel` | `bool` | Defaults to `True`; covered in depth two cells from now |

In [5]:
class MathHomeworkOutput(BaseModel):
    is_math_homework: bool
    reasoning: str


guardrail_agent = Agent(
    name="Guardrail check",
    instructions=(
        "Check if the user is asking you to do their "
        "math homework."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=MathHomeworkOutput,
)


@input_guardrail
async def math_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        guardrail_agent, input, context=ctx.context
    )
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math_homework,
    )


support_agent = Agent(
    name="Customer support agent",
    instructions=(
        "You are a customer support agent. "
        "You help customers with their questions."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[math_guardrail],
)

## Cell 6: Running the Guardrail — Safe and Unsafe Input

This cell runs `support_agent` twice, once with an on-topic message and once
with a math question, to see both sides of the guardrail's behaviour.

The first `try` block sends a harmless customer-support question. The
guardrail's classification agent should return `is_math_homework=False`, the
tripwire never fires, and `support_agent`'s own answer prints normally.

The second `try` block sends an actual math problem. This time the guardrail
should trip, which raises `InputGuardrailTripwireTriggered`. Inside the
`except` block, `e.guardrail_result.guardrail.get_name()` reads the name of
the specific guardrail that fired. Since `math_guardrail` was decorated with
`@input_guardrail` and no explicit `name` was given, `get_name()` falls back
to the function's own name, `"math_guardrail"`. `e.guardrail_result.output.output_info`
is the exact `MathHomeworkOutput` object the guardrail returned, so you can
read its `reasoning` field to see why the classifier made its call.

Run this cell and confirm the second block prints the guardrail's own
reasoning, not just the fact that it tripped.

In [6]:
try:
    result = await Runner.run(
        support_agent,
        "Hello, can you help me track my order?",
    )
    print("Safe input result:", result.final_output)
except InputGuardrailTripwireTriggered:
    print("Unexpectedly blocked")

try:
    result = await Runner.run(
        support_agent,
        "Hello, can you help me solve for x: 2x + 3 = 11?",
    )
    print("Result:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("Guardrail tripped!")
    print(
        f"Guardrail name: "
        f"{e.guardrail_result.guardrail.get_name()}"
    )
    info: MathHomeworkOutput = (
        e.guardrail_result.output.output_info
    )
    print(f"Reasoning: {info.reasoning}")

Safe input result: Absolutely — I can help with that.

Please send me:
- your order number, and
- the email used for the purchase

If you have it, I can also look it up by:
- shipping ZIP/postcode
- tracking number


Guardrail tripped!
Guardrail name: math_guardrail
Reasoning: The user is asking to solve an algebra equation for x, which is a typical math homework problem.


## Cell 7: The run_in_parallel Parameter

`run_in_parallel` is the single most misunderstood parameter in the whole
guardrail system, so this cell isolates it on its own before the timing demo
in the next cell makes the effect visible.

Both guardrails below run the exact same check as `math_guardrail`. The only
difference is the keyword argument passed to `@input_guardrail`:

- **`parallel_check`** uses `run_in_parallel=True`, which is also the
  default you already saw implicitly on `math_guardrail`.
- **`blocking_check`** uses `run_in_parallel=False`.

Straight from the SDK's own docstring: *"Whether the guardrail runs
concurrently with the agent (`True`, default) or before the agent starts
(`False`)."*

In plain terms: `True` means your expensive main agent and your cheap
guardrail check start at the same time. `False` means the guardrail goes
first, full stop, and the main agent never starts until it's given the green
light.

In [7]:
@input_guardrail(run_in_parallel=True)
async def parallel_check(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        guardrail_agent, input, context=ctx.context
    )
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math_homework,
    )


@input_guardrail(run_in_parallel=False)
async def blocking_check(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        guardrail_agent, input, context=ctx.context
    )
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_math_homework,
    )

## Cell 8: Timing Demonstration — Parallel vs Blocking

This cell builds two agents, identical except for which guardrail from the
previous cell they carry, and times a run on each with the same harmless
message.

`parallel_agent` uses `parallel_check` (`run_in_parallel=True`).
`blocking_agent` uses `blocking_check` (`run_in_parallel=False`). Both time
a single `await Runner.run(...)` call with `time.time()` before and after.

There is no universally correct answer between these two modes. If your
guardrail rarely trips, parallel mode costs you almost nothing extra, since
the main agent's call was already running the whole time the guardrail was
checking. If a trip is expensive or dangerous, for example a tool call that
shouldn't fire on bad input, blocking mode is worth the added latency,
because it guarantees the main agent's call never starts until the guardrail
gives the green light. Parallel mode's tradeoff is the reverse: if the
tripwire fires after the main agent's call has already started, those tokens
are spent and wasted. Blocking mode never wastes those tokens, at the cost
of always paying the guardrail's latency up front.

Watch the two elapsed times closely. On a message that doesn't trip the
guardrail, `run_in_parallel=False` should measurably lag behind
`run_in_parallel=True`, since it pays the guardrail's full latency before the
main agent is even allowed to start.

In [8]:
parallel_agent = Agent(
    name="Parallel Guard Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[parallel_check],
)
blocking_agent = Agent(
    name="Blocking Guard Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[blocking_check],
)
msg = "What is the capital of France?"

start = time.time()
result1 = await Runner.run(parallel_agent, msg)
elapsed1 = time.time() - start
print(f"Parallel guardrail: {elapsed1:.1f}s")

start = time.time()
result2 = await Runner.run(blocking_agent, msg)
elapsed2 = time.time() - start
print(f"Blocking guardrail: {elapsed2:.1f}s")

Parallel guardrail: 0.8s
Blocking guardrail: 1.3s


## Cell 9: Multiple Guardrails on One Agent

`input_guardrails` accepts a list, and this cell shows why that matters:
an agent can enforce more than one independent check at the same time.

`profanity_check` is a second, much simpler guardrail. It doesn't call an
agent at all. It converts the input to a string and does a plain substring
check for one blocked word. This is a reminder that a guardrail function can
be as cheap or as elaborate as the check calls for. Nothing about the
`@input_guardrail` decorator requires an LLM call underneath.

`multi_guard_agent` carries both `math_guardrail` and `profanity_check` in
its `input_guardrails` list. All guardrails on an agent run, each respecting
its own `run_in_parallel` setting, and if **any** tripwire fires, the
exception is raised for that specific guardrail. A clean message like the
one below should sail through both checks untouched.

In [9]:
@input_guardrail
async def profanity_check(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    text = input if isinstance(input, str) else str(input)
    has_profanity = "damn" in text.lower()
    return GuardrailFunctionOutput(
        output_info={"checked": True},
        tripwire_triggered=has_profanity,
    )


multi_guard_agent = Agent(
    name="Multi Guard Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    input_guardrails=[math_guardrail, profanity_check],
)
result = await Runner.run(
    multi_guard_agent, "What's the weather like?"
)
print("Passed both guardrails:", result.final_output)

Passed both guardrails: I can help, but I need your location.  
Send me your city, ZIP/postal code, or current location, and I’ll give you the weather.


## Cell 10: RunConfig.input_guardrails — Run-Level Guardrails

Every guardrail so far has been attached to an agent through
`Agent.input_guardrails`. This cell shows the other place a guardrail can
live: `RunConfig.input_guardrails`, passed to `Runner.run(run_config=...)`
instead of to the agent itself.

To make the distinction obvious rather than redundant, this cell attaches
`profanity_check`, not `math_guardrail`, through `RunConfig`. Look back at
`support_agent`'s own `input_guardrails` list in Cell 5: it only contains
`math_guardrail`. `profanity_check` was never attached to `support_agent`
directly. If `RunConfig.input_guardrails` did nothing new, a profane message
would sail straight through here, since none of `support_agent`'s own
guardrails check for profanity.

Instead, watch the second call below trip, even though `support_agent`
itself has no idea `profanity_check` exists. That's a run-level guardrail
doing real work: enforcing a check the agent was never given, on top of
whatever the agent already enforces on its own. `RunConfig.input_guardrails`
supplements `Agent.input_guardrails`, it does not replace it, and this is
what that actually buys you.

In [10]:
try:
    result = await Runner.run(
        support_agent,
        "What's your return policy?",
        run_config=RunConfig(
            workflow_name="Guardrail demo",
            input_guardrails=[profanity_check],
        ),
    )
    print("Safe input result:", result.final_output)
except InputGuardrailTripwireTriggered:
    print("Unexpectedly blocked")

try:
    result = await Runner.run(
        support_agent,
        "This return policy is a damn mess, can you clarify it?",
        run_config=RunConfig(
            workflow_name="Guardrail demo",
            input_guardrails=[profanity_check],
        ),
    )
    print("Result:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("Guardrail tripped!")
    print(
        f"Guardrail name: "
        f"{e.guardrail_result.guardrail.get_name()}"
    )

Safe input result: I can help with that, but I don’t have your store’s specific return policy on hand.

If you share the policy text or your store name/order details, I can explain it clearly. Otherwise, a typical return policy covers:
- return window (e.g., 30 days)
- item condition requirements
- proof of purchase
- refund method and timing
- non-returnable items

If you’d like, I can also help you draft a return policy or a response to send a customer.
Guardrail tripped!
Guardrail name: profanity_check


## Scope Note: Input Guardrails Only Apply to the First Agent

One thing worth remembering from Lecture 5.3: input guardrails run once, at
the very first agent in a chain. If that agent hands off to a specialist
downstream, the specialist does not re-check the input. It already passed.

This is intentional. The check happens once, at the entry point of the
conversation, not on every hop the conversation makes afterward.

## Reference Summary

| Parameter | Default | Effect |
|---|---|---|
| `run_in_parallel=True` | Default | Runs concurrently with the agent. No added latency if it passes, but the main agent's call may already be running when the tripwire fires |
| `run_in_parallel=False` | Opt-in | Must complete before the agent starts. Adds guaranteed latency but zero wasted agent tokens |
| `name` | Function name | Used in tracing and returned by `get_name()` |

| To do this | Use |
|---|---|
| Check input before the agent responds | `@input_guardrail` + `Agent.input_guardrails` |
| Guarantee zero wasted tokens on bad input | `run_in_parallel=False` |
| Minimise added latency when input is usually valid | `run_in_parallel=True` (default) |
| Enforce a check across a workflow, even on agents that don't declare it | `RunConfig.input_guardrails` |
| Read what a guardrail decided after a trip | Catch `InputGuardrailTripwireTriggered`, read `.guardrail_result` |